# Live experiment

Copy this folder, rename it, and work through the cells marked **TODO**. Each
TODO cell lists common choices as comments and ends with a `raise` so you
cannot run the notebook before you have filled it in. Everything else works as
it is.

The [live experiment example](https://github.com/pertzlab/faro/blob/main/examples/02_live_experiment.ipynb) runs this exact structure on a
virtual microscope and explains every step. The [README](https://github.com/pertzlab/faro/blob/main/README.md) is the
reference.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import faro.core.utils as utils
from faro.core.controller import Controller
from faro.core.data_structures import RTMSequence, SegmentationMethod, combine
from faro.core.pipeline import ImageProcessingPipeline
from faro.core.utils import events_to_dataframe
from faro.core.writers import OmeZarrWriter

## TODO: microscope

Fill in this cell, then delete the `raise` line. Pertzlab scopes live in `faro.microscope.pertzlab`; for a new
scope see [Adding Your Own Micro-Manager Microscope](https://github.com/pertzlab/faro/blob/main/README.md#adding-your-own-micro-manager-microscope).

In [ ]:
# from faro.microscope.pertzlab.moench import Moench;   mic = Moench(None)
# from faro.microscope.pertzlab.niesen import Niesen;   mic = Niesen(None)
# from faro.microscope.demo import MMDemo;              mic = MMDemo()
raise NotImplementedError("TODO: choose a microscope")

## Open napari

Live view, stage control and the position list come from napari-micromanager.
The status widget shows the run once a controller is bound to it below.

In [ ]:
import napari
from napari_micromanager import MainWindow
from faro.widgets import ExperimentStatusWidget

viewer = napari.Viewer()
mm_widget = MainWindow(viewer, mmcore=mic.mmc)
viewer.window.add_dock_widget(mm_widget, name="napari-micromanager")

status_widget = ExperimentStatusWidget()
viewer.window.add_dock_widget(status_widget, name="experiment status", area="right")

## TODO: experiment settings

Fill in this cell, then delete the `raise` line. Channel `config` names must exist in your Micro-Manager channel
group. Use `PowerChannel` when the stimulation channel has a power setting.

In [ ]:
STORAGE_ROOT = r"D:\data"                 # results go to STORAGE_ROOT/EXPERIMENT_NAME
EXPERIMENT_NAME = "2026-01-01_my_experiment"

INTERVAL_S = 30                            # seconds between frames
N_BASELINE = 20                            # frames before stimulation
N_STIM = 10                                # frames with stimulation
N_RECOVERY = 60                            # frames after stimulation

IMAGING_CHANNELS = [{"config": "miRFP", "exposure": 300}]
STIM_CHANNEL = {"config": "CyanStim", "exposure": 200}
# from faro.core.data_structures import PowerChannel
# STIM_CHANNEL = PowerChannel(config="CyanStim", exposure=200, power=10)
raise NotImplementedError("TODO: set the experiment settings")

## TODO: segmentation

Fill in this cell, then delete the `raise` line. Every `SegmentationMethod` needs a name; the feature extractor and
stimulator refer to it. Cellpose needs `uv sync --extra cellpose`.

In [ ]:
# from faro.segmentation.base import OtsuSegmentator;        seg = OtsuSegmentator()
# from faro.segmentation.cellpose_v4 import CellposeV4;      seg = CellposeV4(diameter=30)
segmentators = [
    SegmentationMethod(name="labels", segmentation_class=seg, use_channel=0, save_tracked=True),
]
raise NotImplementedError("TODO: choose a segmentator")

## TODO: feature extraction

Fill in this cell, then delete the `raise` line. The extractor returns one row per label. Subclass `FeatureExtractor`
for anything beyond the built-ins, see
[Writing your own components](https://github.com/pertzlab/faro/blob/main/README.md#writing-your-own-components).

In [ ]:
# from faro.feature_extraction.simple import SimpleFE;       feature_extractor = SimpleFE("labels")
# from faro.feature_extraction.erk_ktr import FE_ErkKtr;     feature_extractor = FE_ErkKtr("labels")
raise NotImplementedError("TODO: choose a feature extractor")

## TODO: stimulation

Fill in this cell, then delete the `raise` line. Set `stimulator = None` for an experiment without feedback. Built-in
stimulators live in `faro.stimulation`; for your own, subclass `Stim`,
`StimWithImage` or `StimWithPipeline`.

In [ ]:
# stimulator = None
# from faro.stimulation.base import StimWholeFOV;                    stimulator = StimWholeFOV()
# from faro.stimulation.percentage_of_cell import StimPercentageOfCell
raise NotImplementedError("TODO: choose a stimulator")

## Pipeline and controller

The tracker's `search_range` is the largest centroid movement between frames
in pixels. Adjust it to your cells and interval.

In [ ]:
from faro.tracking.trackpy import TrackerTrackpy

path = os.path.join(STORAGE_ROOT, EXPERIMENT_NAME)
os.makedirs(path, exist_ok=True)

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=TrackerTrackpy(search_range=30),
    stimulator=stimulator,
)
ctrl = Controller(mic, pipeline, writer=OmeZarrWriter(storage_path=path))
print("results go to", path)

In [ ]:
status_widget.set_controller(ctrl)

## DMD calibration

Scopes with a DMD need a fresh calibration every session so stimulation masks land on the right pixels. `calibrate_dmd` always runs when called and does nothing on a scope without a DMD. The calibration light hits your sample, so move to an empty area first.

In [ ]:
if stimulator is not None:
    mic.calibrate_dmd(STIM_CHANNEL["config"])

## Preview

Snap one frame at the current position and check labels and stimulation mask
before committing to a long run.

In [ ]:
channel = IMAGING_CHANNELS[0]["config"]
mic.mmc.setConfig(mic.resolve_group(channel), channel)
mic.mmc.snapImage()
test_img = mic.mmc.getImage()
labels_preview = segmentators[0].segmentation_class.segment(test_img)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_img, cmap="gray")
axes[0].set_title("Raw image")
axes[1].imshow(labels_preview, cmap="nipy_spectral")
axes[1].set_title(f"Labels ({labels_preview.max()} cells)")
axes[2].imshow(test_img, cmap="gray")
if stimulator is not None:
    from faro.core.pipeline import dispatch_stim_mask
    mask_preview = dispatch_stim_mask(
        stimulator, {"labels": labels_preview}, {"img_shape": test_img.shape}, img=test_img[None]
    )
    if mask_preview is True:                       # StimWholeFOV
        mask_preview = np.ones_like(labels_preview)
    axes[2].imshow(np.ma.masked_where(mask_preview == 0, mask_preview), cmap="autumn", alpha=0.8)
axes[2].set_title("Stimulation mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## TODO: positions

Fill in this cell, then delete the `raise` line. Pick positions in the napari-micromanager MDA widget and read them
with `generate_fov_positions(mic, viewer=viewer)`, or list them by hand.

In [ ]:
# fov_positions = utils.generate_fov_positions(mic, viewer=viewer)
# fov_positions = utils.generate_fov_positions_from_list(mic, [{"x": 0.0, "y": 0.0, "z": 0.0}])
raise NotImplementedError("TODO: define the positions")

## Build the event list

Baseline, stimulation and recovery phases joined in time. Delete or add phases
as needed; `rtm_metadata` ends up in the tracks so you can group by phase.

In [ ]:
baseline = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_BASELINE},
    stage_positions=fov_positions,
    channels=IMAGING_CHANNELS,
    rtm_metadata={"phase": "baseline"},
)
stim_phase = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_STIM},
    stage_positions=fov_positions,
    channels=IMAGING_CHANNELS,
    stim_channels=[STIM_CHANNEL] if stimulator is not None else None,
    stim_frames=range(N_STIM) if stimulator is not None else None,
    rtm_metadata={"phase": "stimulation"},
)
recovery = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_RECOVERY},
    stage_positions=fov_positions,
    channels=IMAGING_CHANNELS,
    rtm_metadata={"phase": "recovery"},
)
events = combine(baseline, stim_phase, recovery, axis="t")
df_events = events_to_dataframe(events)
print(f"{len(events)} events over {df_events['time'].max() / 60:.1f} min")
df_events.head()

## Validate and load

Fix every warning before you start. Validation checks the pipeline components, the event metadata, channel names, exposure limits and the DMD calibration. `load_experiment` then shows the plan in the status widget.

In [ ]:
assert ctrl.validate_events(events), "fix the warnings above before running"
ctrl.load_experiment(events, stim_mode="current")

## Run

Start from the widget or with the next cell. The kernel stays free during the run.

In [ ]:
handle = ctrl.start_experiment()

In [ ]:
handle.status()

## Finish

Blocks until the last frame is processed, then writes `exp_data.parquet` with
all fields of view combined.

In [ ]:
final = handle.wait()
ctrl.finish_experiment()
mic.post_experiment()
utils.generate_exp_data_from_tracks(path)
print(f"state {final.state!r}, {final.n_frames_received} frames, "
      f"{len(final.background_errors)} background errors")
pd.read_parquet(os.path.join(path, "exp_data.parquet")).head()